In [1]:
import pandas as pd
import numpy as np
import os

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_generator_bearings_univariate\univariate\dsas_g11_generator_bearings_univariate_output4.xlsx'


# سنسورهایی که تست ۳-سیگما روی آن‌ها اجرا می‌شود
target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']

def run_daily_weekly_analysis():
    if not os.path.exists(file_path):
        print(f"❌ خطا: فایل در مسیر زیر یافت نشد:\n{file_path}")
        return

    try:
        # ۱. بارگذاری داده‌ها
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print("✅ مرحله ۱: داده‌ها بارگذاری و بر اساس زمان مرتب شدند.")
    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")
        return

    # ۲. جداسازی بازه یک ماه اخیر برای تحلیل
    last_date = df['date'].max()
    one_month_ago = last_date - pd.Timedelta(days=30)
    df_recent = df[df['date'] >= one_month_ago].copy()

    # ۳. محاسبات Dual EWMA (روزانه در مقابل هفتگی)
    # با فرض ۶ داده در روز:
    # بازه سریع (روزانه): 6 داده -> alpha = 2 / (6 + 1) ≈ 0.28
    # بازه کند (هفتگی): 42 داده (6*7) -> alpha = 2 / (42 + 1) ≈ 0.046

    alpha_fast = 0.22  # حساس به تغییرات ۲۴ ساعت اخیر
    alpha_slow = 0.035 # نمایانگر روند کلی ۷ روز اخیر

    results_list = []

    print(f"⏳ تحلیل روند: سریع (Alpha={alpha_fast}) و کند (Alpha={alpha_slow}) در حال انجام است...")

    for col in target_sensors:
        if col in df_recent.columns:
            # ایجاد دیتافریم موقت
            temp_df = df_recent[['date', col]].copy()
            temp_df = temp_df.rename(columns={col: 'Raw_Value'})
            temp_df['AssetID'] = col

            # الف) میانگین متحرک سریع (بازه یک روزه)
            # منعکس‌کننده نوسانات روز جاری
            temp_df['Daily_EWMA_Fast'] = temp_df['Raw_Value'].ewm(alpha=alpha_fast, adjust=False).mean()

            # ب) میانگین متحرک کند (بازه هفتگی)
            # فیلتر کردن نویزها و مشخص کردن وضعیت سلامت پایدار تجهیز
            temp_df['Weekly_EWMA_Slow'] = temp_df['Raw_Value'].ewm(alpha=alpha_slow, adjust=False).mean()

            # ج) محاسبه انحراف سیگنال (Gap)
            # فاصله گرفتن این دو میانگین به معنی تغییر رفتار فیزیکی است
            temp_df['Signal_Gap'] = temp_df['Daily_EWMA_Fast'] - temp_df['Weekly_EWMA_Slow']

            # د) محاسبه درصد انحراف نسبت به روند هفتگی (اختیاری اما بسیار کاربردی)
            temp_df['Deviation_Percent'] = (temp_df['Signal_Gap'] / temp_df['Weekly_EWMA_Slow']) * 100

            # انتخاب و چیدمان ستون‌ها
            cols_order = ['date', 'AssetID', 'Raw_Value', 'Daily_EWMA_Fast', 'Weekly_EWMA_Slow', 'Signal_Gap', 'Deviation_Percent']
            results_list.append(temp_df[cols_order])

    if not results_list:
        print("⚠️ هشدار: سنسورهای مورد نظر در فایل یافت نشدند.")
        return

    # ۴. تجمیع نتایج

    df_final_output = pd.concat(results_list, ignore_index=True)

    # ۵. ذخیره در اکسل (خروجی شماره ۴)
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename) as writer:
            df_final_output.to_excel(writer, index=False, sheet_name='Maintenance_Strategy')

        print(f"🚀 گزارش هوشمند با رویکرد روزانه/هفتگی ساخته شد.")
        print(f"📍 مسیر فایل: {output_filename}")
        print("-"*50)
        print("💡 راهنمای تحلیل:")
        print("1. ستون Daily_EWMA_Fast: وضعیت لرزش/دما در حدوداً ۶ داده اخیر (امروز).")
        print("2. ستون Weekly_EWMA_Slow: روند کلی تجهیز در یک هفته اخیر.")
        print("3. اگر Deviation_Percent رشد مداوم داشته باشد، احتمالا خرابی در حال شکل‌گیری است.")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل اکسل: {e}")

# اجرای نهایی
run_daily_weekly_analysis()

✅ مرحله ۱: داده‌ها بارگذاری و بر اساس زمان مرتب شدند.
⏳ تحلیل روند: سریع (Alpha=0.22) و کند (Alpha=0.035) در حال انجام است...
🚀 گزارش هوشمند با رویکرد روزانه/هفتگی ساخته شد.
📍 مسیر فایل: outputs\G11\dsas_g11_generator_bearings_univariate\univariate\dsas_g11_generator_bearings_univariate_output4.xlsx
--------------------------------------------------
💡 راهنمای تحلیل:
1. ستون Daily_EWMA_Fast: وضعیت لرزش/دما در حدوداً ۶ داده اخیر (امروز).
2. ستون Weekly_EWMA_Slow: روند کلی تجهیز در یک هفته اخیر.
3. اگر Deviation_Percent رشد مداوم داشته باشد، احتمالا خرابی در حال شکل‌گیری است.
